In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("twibot")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .getOrCreate())
print(spark.version)

In [ ]:
import os
base = "/home/jovyan/work/data/twibot_dataset"
out  = "/home/jovyan/work/parquet"

for name in ["label", "split"]:
    try:
        df = spark.read.csv(f"{base}/{name}.csv", header=True)
        df.write.mode("overwrite").parquet(f"{out}/{name}")
        print(name, "done")
    except Exception as e:
        print(name, "FAILED")
        print(str(e)[:1500])
        break

In [ ]:
df = spark.read.csv(f"{base}/edge.csv", header=True)
df.write.mode("overwrite").parquet(f"{out}/edge")
print("edge done")

In [7]:
import shutil
shutil.rmtree("/home/jovyan/work/parquet/edge", ignore_errors=True)

In [ ]:
base = "/home/jovyan/work/data/twibot_dataset"
out  = "/home/jovyan/work/parquet"

df = spark.read.csv(f"{base}/edge.csv", header=True).limit(5_000_000)
df.write.mode("overwrite").parquet(f"{out}/edge")
print("edge (5M rows) done")

In [ ]:
from pyspark.sql import functions as F

edges = spark.read.parquet(f"{out}/edge")
print("rows:", edges.count())
edges.groupBy("relation").count().orderBy("count", ascending=False).show(30, False)
print("distinct sources:", edges.select("source_id").distinct().count())

In [ ]:
from pyspark.sql import functions as F

edges  = spark.read.parquet(f"{out}/edge")
labels = spark.read.parquet(f"{out}/label")
ids = labels.select("id")

follow = (edges.filter(F.col("relation").isin("following", "followers"))
               .select("source_id", "target_id"))

inside = (follow
    .join(ids.withColumnRenamed("id", "source_id"), "source_id")
    .join(ids.withColumnRenamed("id", "target_id"), "target_id")
    .dropDuplicates())

n_inside = inside.count()
print("follow edges:", follow.count(), "| with both ends labeled:", n_inside)

In [ ]:

g_edges = inside.select(F.col("source_id").alias("src"),
                        F.col("target_id").alias("dst")).dropDuplicates()
g_edges.write.mode("overwrite").parquet(f"{out}/graph_edges")

# vertices: only users that appear in at least one edge
v_ids = (g_edges.select(F.col("src").alias("id"))
         .union(g_edges.select(F.col("dst").alias("id"))).distinct())
verts = v_ids.join(labels, "id")
verts.write.mode("overwrite").parquet(f"{out}/graph_vertices")

verts = spark.read.parquet(f"{out}/graph_vertices")
g_edges = spark.read.parquet(f"{out}/graph_edges")
print("vertices:", verts.count(), "| edges:", g_edges.count())
verts.groupBy("label").count().show()

In [ ]:
!pip install graphframes

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName("twibot")
    .config("spark.jars.packages",
            "graphframes:graphframes:0.8.3-spark3.5-s_2.12,"
            "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0")
    .config("spark.mongodb.write.connection.uri", "mongodb://mongo:27017")
    .config("spark.mongodb.read.connection.uri", "mongodb://mongo:27017")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate())
spark.sparkContext.setCheckpointDir("/tmp/ckpt")
out = "/home/jovyan/work/parquet"
print(spark.version)

3.5.3


In [ ]:
from graphframes import GraphFrame

verts = (spark.read.parquet(f"{out}/graph_vertices")
         .withColumnRenamed("label", "bot_label"))
edges = spark.read.parquet(f"{out}/graph_edges")

deg = (edges.select(F.col("src").alias("id"))
       .union(edges.select(F.col("dst").alias("id")))
       .groupBy("id").count())
top = deg.orderBy(F.desc("count")).limit(50000).select("id")

v = verts.join(top, "id")
e = (edges.join(top.withColumnRenamed("id", "src"), "src")
          .join(top.withColumnRenamed("id", "dst"), "dst"))
v.cache(); e.cache()
print("vertices:", v.count(), "| edges:", e.count())
v.groupBy("bot_label").count().show()

In [ ]:
g = GraphFrame(v, e)

lp = g.labelPropagation(maxIter=5)   # adds a "label" column = community id

res = lp.groupBy("label").agg(
    F.count("*").alias("size"),
    F.round(F.avg((F.col("bot_label") == "bot").cast("int")), 3).alias("bot_ratio"))
res.orderBy(F.desc("size")).show(15)
print("communities:", res.count())

In [ ]:
conf = spark.sparkContext.getConf()
print("packages:", conf.get("spark.jars.packages", "NOT SET"))
print("jars:", conf.get("spark.jars", "NOT SET"))

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName("twibot")
    .config("spark.jars.packages",
            "graphframes:graphframes:0.8.3-spark3.5-s_2.12,"
            "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate())
spark.sparkContext.setCheckpointDir("/tmp/ckpt")
out = "/home/jovyan/work/parquet"

conf = spark.sparkContext.getConf()
print("packages:", conf.get("spark.jars.packages", "NOT SET"))
print("jars:", conf.get("spark.jars", "NOT SET"))

In [ ]:
from graphframes import GraphFrame

verts = (spark.read.parquet(f"{out}/graph_vertices")
         .withColumnRenamed("label", "bot_label"))
edges = spark.read.parquet(f"{out}/graph_edges")

deg = (edges.select(F.col("src").alias("id"))
       .union(edges.select(F.col("dst").alias("id")))
       .groupBy("id").count())
top = deg.orderBy(F.desc("count")).limit(50000).select("id")

v = verts.join(top, "id")
e = (edges.join(top.withColumnRenamed("id", "src"), "src")
          .join(top.withColumnRenamed("id", "dst"), "dst"))
v.cache(); e.cache()
print("vertices:", v.count(), "| edges:", e.count())

g = GraphFrame(v, e)
print("graph built")

In [ ]:
from graphframes import GraphFrame

verts = (spark.read.parquet(f"{out}/graph_vertices")
         .withColumnRenamed("label", "bot_label"))
edges = spark.read.parquet(f"{out}/graph_edges")

deg = (edges.select(F.col("src").alias("id"))
       .union(edges.select(F.col("dst").alias("id")))
       .groupBy("id").count())
top = deg.orderBy(F.desc("count")).limit(50000).select("id")

v = verts.join(top, "id")
e = (edges.join(top.withColumnRenamed("id", "src"), "src")
          .join(top.withColumnRenamed("id", "dst"), "dst"))
v.cache(); e.cache()
print("vertices:", v.count(), "| edges:", e.count())

g = GraphFrame(v, e)
lp = g.labelPropagation(maxIter=5)
lp.write.mode("overwrite").parquet(f"{out}/communities")
lp = spark.read.parquet(f"{out}/communities")

res = lp.groupBy("label").agg(
    F.count("*").alias("size"),
    F.round(F.avg((F.col("bot_label") == "bot").cast("int")), 3).alias("bot_ratio"))
res.orderBy(F.desc("size")).show(5)
print("communities:", res.count())

In [ ]:
base_rate = 3118 / 50000

lab = lp.select("id", F.col("label").alias("community"))
und = (e.select(F.least("src", "dst").alias("a"),
                F.greatest("src", "dst").alias("b")).dropDuplicates())

inner = (und
    .join(lab.withColumnRenamed("id", "a").withColumnRenamed("community", "ca"), "a")
    .join(lab.withColumnRenamed("id", "b").withColumnRenamed("community", "cb"), "b")
    .filter("ca = cb").groupBy("ca").count()
    .withColumnRenamed("count", "inner_edges"))

stats = (res.join(inner, res.label == inner.ca, "left").drop("ca")
    .withColumn("density", F.round(F.col("inner_edges") / (F.col("size") * (F.col("size") - 1) / 2), 5))
    .withColumn("lift", F.round(F.col("bot_ratio") / F.lit(base_rate), 2)))

stats.filter("size >= 50").orderBy(F.desc("lift")).show(20)

In [ ]:
stats = (res.join(inner, res.label == inner.ca, "left").drop("ca")
    .fillna(0, subset=["inner_edges"])
    .withColumn("density", F.round(F.col("inner_edges") / (F.col("size") * (F.col("size") - 1) / 2), 5))
    .withColumn("lift", F.round(F.col("bot_ratio") / F.lit(base_rate), 2))
    .withColumn("is_connected", F.col("inner_edges") >= F.col("size")))   # at least ~1 link per member on average

stats.write.mode("overwrite").parquet(f"{out}/community_stats")
stats.filter("size >= 50 AND is_connected").orderBy(F.desc("lift")).show(15)

In [ ]:
with open("/home/jovyan/work/data/twibot_dataset/user.json") as fh:
    print(fh.read(1500))
    